# E6 — Calendar Spread Intraday W2/W3/W4

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
from matplotlib import cm
plasma = cm.get_cmap('plasma')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, wk in zip(axes, ['W2','W3','W4']):
    wm = WINDOWS_META[wk]
    colors_day = [plasma(i/6) for i in range(7)]
    for di, (day, lbl, dc) in enumerate(zip(wm['days'], wm['day_labels'], colors_day)):
        mids = {}
        for sym in [wm['front'], wm['back']]:
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            if not fpath: continue
            df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
            df['ts_event'] = pd.to_datetime(df['ts_event'])
            rth = ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) >= wm['rth_start_min']) &                   ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) <= wm['rth_end_min'])
            df = df[rth].set_index('ts_event')
            mids[sym] = ((df['bid_px_00']+df['ask_px_00'])/2).resample('1min').last().dropna()
            del df; gc.collect()
        if wm['front'] not in mids or wm['back'] not in mids:
            continue
        cal = mids[wm['back']].subtract(mids[wm['front']], fill_value=np.nan).dropna()
        t0 = cal.index.normalize()
        mins = (cal.index - t0).seconds / 60
        ax.plot(mins, cal.values, color=dc, lw=0.8, alpha=0.8, label=lbl)
    ax.set_xlabel('Minutes from midnight UTC')
    ax.set_ylabel('Calendar Spread (pts)')
    ax.set_title(wk)
    ax.legend(fontsize=6)
fig.suptitle('Calendar Spread Intraday — W2/W3/W4 (D1→D7, plasma colormap)', fontsize=13)
save_fig(fig, 'E6_cal_spread_intraday_w2w3w4.png')
